In [1]:
import requests
import time
import json
from tqdm import *
import pandas as pd
import os
import sys
sys.path.append("../../source/")
from concurrent.futures import ThreadPoolExecutor, as_completed
from llm_agent import LLMClient
from IntentAgent import IntentAgent
from RewriteQueryAgent import RewriteQueryAgent
from RagEvidenceExtractAgent import RagEvidenceExtractAgent
from CompleteRagAgent import CompleteRagAgent
from creat_class_from_config import create_class_from_config
from WebRetriever import web_retriever
from WebSummaryAgent import WebSummaryAgent

import logging

# 基础配置
logging.basicConfig(
    level=logging.DEBUG,  # 打印级别：DEBUG < INFO < WARNING < ERROR < CRITICAL
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
# 获取日志对象
logger = logging.getLogger("deepagent")

# from dag import DAG, DAGNode
def read_conf(path):
    with open(path) as f:
        agent_prompt = json.load(f)
        return agent_prompt
    
def edit_distance(s1: str, s2: str) -> int:
    """空间优化版编辑距离"""
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    
    m, n = len(s1), len(s2)
    prev = list(range(n + 1))
    
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    
    return prev[n]

def longest_common_subsequence(s1: str, s2: str) -> int:
    """最长公共子序列长度"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    
    return dp[m][n]

def same_ratio(s1: str, s2: str) -> dict:
    """计算两种相同成分占比"""
    if s1 == s2:
        return {"lcs_ratio": 1.0, "edit_ratio": 1.0}
    if not s1 or not s2:
        return {"lcs_ratio": 0.0, "edit_ratio": 0.0}
    
    lcs_len = longest_common_subsequence(s1, s2)
    edit_dist = edit_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    
    return {
        "lcs_ratio": lcs_len / max_len,
        "edit_ratio": (max_len - edit_dist) / max_len,
        "lcs_length": lcs_len,
        "edit_distance": edit_dist,
        "max_length": max_len
    }

/home/work/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.8' currently installed).
  warnings.warn(msg, UserWarning)


In [2]:
def intent_router(memory):
    if "经典句子" in memory.get_intent():
        return "RAG"
    elif "工作写作" in memory.get_intent():
        return "DeepAgent"
    else:
        return 'create'

In [3]:
def generate_result(outpath, query):
    prompts = read_conf("agent_prompt.json")
    llm_client = LLMClient()
    MemerySystem = create_class_from_config("agent_class.json")
    memory = MemerySystem()
    memory.set_query(query)
#     intent_llm = IntentAgent(prompt=prompts["intent"], doc="intent")
#     intent_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "意图识别  " + "".join(memory.get_raise_error()))
    sence = intent_router(memory)
    print("sence", sence)
    
    ###  rag
    rewrite_llm = RewriteQueryAgent(prompt=prompts["rewrite"], doc="rewrite")
    rewrite_llm.exceute(llm_client, memory)
    if memory.get_need_search() == "0":
        print("need_search", memory.get_need_search())
        return memory
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "query改写  " + "".join(memory.get_raise_error()))
        return memory
    all_result = {}
    tmp_value = []
    for query in memory.get_search_query():
        result = web_retriever(query)
        for k, v in result.items():
            need = True
            for tv in tmp_value:
                srtv = same_ratio(v, tv)
                if srtv["edit_ratio"] > 0.66:
                    need = False
                    break
            tmp_value.append(v)
            if need:
                all_result[query +"_" + k] = v
    memory.set_evdences(json.dumps(all_result, ensure_ascii=False))
    ragextract_llm = RagEvidenceExtractAgent(prompt=prompts["rag_extract"], doc="extract")
    ragextract_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "rag extract " + "".join(memory.get_raise_error()))
        return memory
    comrag_llm = CompleteRagAgent(prompt=prompts["rag_complete"], doc="complete")
    comrag_llm = comrag_llm.exceute(llm_client, memory)
    if len(memory.get_raise_error()) >= 2:
        logger.error("错误信息: " + "rag complete " + "".join(memory.get_raise_error()))
        return memory
    return memory

def main(inpath, outpath):
    remaining_queries = []
    dataframe = pd.read_excel(inpath)
    for indx, row in dataframe.iterrows():
        query = row['Query']
        remaining_queries.append(query)
    print(remaining_queries[1])
    print(remaining_queries[2])
    with open(outpath, 'a') as f:
        with ThreadPoolExecutor(max_workers=2) as executor:
            # 提交任务给线程池
            futures = {executor.submit(generate_result,outpath, query_prompt): query_prompt for query_prompt in remaining_queries}
            for future in tqdm(as_completed(futures), total=len(remaining_queries)):
                try:
                    memory = future.result()
                    # 立即将结果写入文件
                    f.write(json.dumps(memory.to_dict(), ensure_ascii=False) + '\n')
                    # f.flush()
                except Exception as e:
                    print(f" 产生了一个异常: {e}")

In [ ]:
if __name__ == "__main__":
    inpath = "DeepAgent/data/RAG/query_RAG检索.xlsx"
    outpath = "DeepAgent/data/RAG/query_RAG检索_result.json"
    main(inpath, outpath)